# Stat 220, Unit 3 Homework: Linear Regression

`housing_data.csv` is 1,000 home sales with `house_price`, `square_footage`,
`num_bedrooms`, `has_garage` (1 or 0), and `neighborhood`.

```python
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

rng = np.random.default_rng(220)
homes = pd.read_csv("https://richardson.byu.edu/220/housing_data.csv")
homes.head()
```

**Problem 1.** *A slope you already know the answer to.* You set the truth, then see what the
regression reports back.

Part a. Run this. The true slope is 1.5. Report the estimate and its 95% interval, and say whether the interval covers the truth.

In [ ]:
n = 200
x = rng.normal(0, 1, n)
y = 3 + 1.5*x + rng.normal(0, 2, n)

fit = smf.ols("y ~ x", data=pd.DataFrame({"x": x, "y": y})).fit()
print(f"estimated slope : {fit.params['x']:.3f}")
print(f"standard error  : {fit.bse['x']:.3f}")
print("95% interval    :", fit.conf_int().loc["x"].round(3).tolist())

The estimated slope is 1.771, with a 95% confidence interval of (1.491, 2.050). The interval contains the true slope of 1.5.


Part b. This repeats that study 500 times. Compare the spread of the 500 estimates to the standard error one study reported in part a, and say what a standard error is measuring.

In [ ]:
slopes = []
for _ in range(500):
    xs = rng.normal(0, 1, n)
    ys = 3 + 1.5*xs + rng.normal(0, 2, n)
    slopes.append(smf.ols("y ~ x", data=pd.DataFrame({"x": xs, "y": ys})).fit().params["x"])

print(f"mean of the 500 estimates : {np.mean(slopes):.3f}")
print(f"SD of the 500 estimates   : {np.std(slopes):.3f}")

The 500 slope estimates have a standard deviation of about 0.143, which is almost the same as the standard error of 0.142 from part a. The standard error measures the typical amount that the estimated slope varies from sample to sample.

Part c. In part b the estimates are centered on 1.5 but individually off by a fair amount. A classmate says the regression is therefore unreliable. Answer them in two or three sentences.

The regression is not unreliable just because individual estimates differ from 1.5. The estimates are centered around the true slope, and the standard error tells us how much we expect estimates to vary from sample to sample.

**Problem 2.** *What is a square foot worth?* A homeowner is deciding whether to add 400 square
feet and wants a number.

Part a. Fit `house_price` on `square_footage` and report the slope with its units and its 95% confidence interval. (`smf.ols("house_price ~ square_footage", data=homes).fit()`, then `.conf_int()`.)

In [1]:
fit2 = smf.ols("house_price ~ square_footage", data=homes).fit()
print(fit2.params)
print(fit2.conf_int())

NameError: name 'smf' is not defined

The estimated slope is **$180.20 per square foot**, with a **95% confidence interval of ($162.39, $198.01) per square foot**. The homes in the data range from about **595 to 3,215 square feet**.


Part b. Write the one sentence you would tell the homeowner, including the units and the range of sizes the data actually covers.

Based on these homes, each additional square foot is associated with about **$180 higher predicted house price**, with the data covering homes from about **595 to 3,215 square feet**.


Part c. Here is what the model missed. Say whether you see a pattern, and what that means for trusting the line.

In [ ]:
fit2 = smf.ols("house_price ~ square_footage", data=homes).fit()
plt.scatter(fit2.fittedvalues, fit2.resid, s=10, alpha=0.5)
plt.axhline(0, color="red"); plt.xlabel("fitted price"); plt.ylabel("residual"); plt.show()

print(f"R-squared   : {fit2.rsquared:.3f}")
print(f"residual SD : {np.sqrt(fit2.scale):,.0f} dollars")

The residuals are scattered around zero without a clear pattern, so there does not appear to be a major problem with using a linear model. The model explains about **80.4% of the variation** in house prices, with a residual SD of about **$41,923**.


Part d. The homeowner asks for the predicted price of a 6,000 square foot house. Look at the range in your part a answer and say, in two or three sentences, what you would tell them.

I would not trust the prediction for a 6,000-square-foot house. The data only covers homes up to about **3,215 square feet**, so 6,000 square feet is a large extrapolation; although the model predicts about **$1.20 million**, we do not have data supporting that prediction.


**Problem 3.** *Adding a category.* The same homes, now with a garage and a neighborhood.

Part a. Report the coefficient on `has_garage` and write the sentence that interprets it. Say exactly what is being held fixed.

In [ ]:
g = smf.ols("house_price ~ square_footage + has_garage", data=homes).fit()
print(g.params.round(1))
print("\n95% interval on has_garage:", g.conf_int().loc["has_garage"].round(0).tolist())

The coefficient on `has_garage` is about **$23,462**. Holding **square footage fixed**, homes with a garage are associated with about **$23,462 higher predicted house prices** than homes without a garage.


Part b. `neighborhood` has several levels. Name the baseline level, and explain how to read one of the other coefficients.

In [ ]:
nb_fit = smf.ols("house_price ~ square_footage + C(neighborhood)", data=homes).fit()
print(nb_fit.params.round(1))
print("\nlevels in the data:", sorted(homes.neighborhood.unique()))

The baseline neighborhood is **Beachside**. For example, the Countryside coefficient of about **−$64,506** means that, holding square footage fixed, homes in Countryside have about **$64,506 lower predicted prices** than comparable homes in Beachside.


Part c. A realtor asks what a garage is worth. Answer in two or three sentences, using the number, its uncertainty, and the homes it applies to.

In these homes, having a garage is associated with about **$23,462 higher predicted price**, holding square footage fixed, with a 95% confidence interval of about **$7,115 to $39,808**. This applies to the homes in our observational dataset, so it describes an association rather than proving that adding a garage would cause the price to increase by that amount.


**Problem 4.** *A coefficient that changes its mind.* Does an extra bedroom raise the price?

Part a. Report the bedroom coefficient from each model and say how much of the first one survived.

In [ ]:
alone = smf.ols("house_price ~ num_bedrooms", data=homes).fit()
with_size = smf.ols("house_price ~ num_bedrooms + square_footage", data=homes).fit()

print(f"bedrooms alone          : {alone.params['num_bedrooms']:+,.0f} dollars per bedroom")
print(f"bedrooms, with size in  : {with_size.params['num_bedrooms']:+,.0f} dollars per bedroom")
print(f"\ncorr(bedrooms, square footage) = "
      f"{homes.num_bedrooms.corr(homes.square_footage):+.2f}")

The bedroom coefficient is about **+$52,814 per bedroom** when bedrooms are used alone, and about **+$23,499 per bedroom** when square footage is also included. About **44.5%** of the original coefficient remains.


Part b. Explain the mechanism in two or three sentences: why does leaving square footage out change the bedroom coefficient?

Bedrooms and square footage are related, so when square footage is left out, the bedroom coefficient partly captures the effect of having a larger house. Adding square footage controls for house size, causing the bedroom coefficient to drop from about **$52,814 to $23,499** per bedroom.


Part c. A builder asks whether splitting the same floor space into more bedrooms would raise the price. Which of the two numbers is closer to an answer for them, and what would still worry you?

The **$23,499** coefficient is closer to answering the builder's question because it compares bedrooms while holding square footage fixed. However, the data are observational, so the coefficient shows an association and does not prove that adding a bedroom would cause the price to increase by $23,499.


**Problem 5.** *What can and cannot be said.* No computer.

Part a. For your Problem 2 square footage slope, write one claim the analysis supports and one that sounds similar but it does not. Say what separates them.

The analysis supports the claim that **larger homes are associated with higher house prices**, at about $180 more per additional square foot. It does not support the claim that **adding a square foot causes the price to increase by $180**, because the data are observational rather than from a randomized experiment.


Part b. Does the neighborhood coefficient in Problem 3 tell you what would happen to a house's price if you picked it up and moved it to that neighborhood? Explain.

No. The neighborhood coefficient tells us how house prices are associated with neighborhood, holding the other predictors in the model fixed; it does not tell us what would happen if the same house were physically moved to another neighborhood. Because the data are observational, other differences between neighborhoods could also explain part of the price difference.


Part c. A colleague summarizes the assignment as "we found that square footage causes price to rise, garages add value, and bedrooms do not matter." Rewrite it so every claim is one your analyses support.

In these homes, **greater square footage is associated with higher house prices, having a garage is associated with higher predicted prices after holding square footage fixed, and bedrooms are positively associated with price, although the bedroom association is smaller after controlling for square footage**. Because these are observational data, these results describe associations rather than proving that changing square footage, adding a garage, or adding a bedroom would cause a particular change in price.
